# Chapter 7.2. 적격흔적(TD(\lambda))과 파라미터 튜닝 실습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter07_2_eligibility_trace_tuning.ipynb)

책 본문: [7.2 적격흔적과 파라미터 튜닝 실습](https://smhanlab.com/book-ml/kor/ml2/chapter07/2.html)

이 노트북은 책 7.2절의 두 가지를 실제로 실행해 봅니다.

1. **n을 바꿔가며** n-step TD의 학습 오차를 20회 반복 평균으로 재고, 편향-분산
   트레이드오프가 만들어내는 **U자 곡선**을 확인합니다.
2. **\(\lambda\)를 바꿔가며** 적격흔적(TD(\(\lambda\)))의 오차를 재고,
   \(\lambda\)를 "모든 n을 동시에" 다루는 다이얼로 튜닝합니다.

그리고 "단 한 번의 실행(단일 시드)으로 최적 지점을 고르면 안 된다"는
실습의 전제를 직접 보여줍니다. 전부는 순수 Python — numpy/matplotlib만 씁니다.

## 1. 환경: 7-상태 random walk (참값이 정확히 알려진 환경)

상태 0~6이 일렬로 늘어서 있고, 0과 6은 종료 상태입니다. 매 스텝 50:50으로
왼쪽/오른쪽으로 이동하며, 오른쪽 끝(6)에 도달하면 보상 \(+1\), 왼쪽 끝(0)에
도달하면 보상 \(0\)을 줍니다. 이 환경은 **참값이 정확히 알려져 있다**는
특징이 있어, 학습한 \(V\)의 오차를객관적으로 재기 좋습니다.

\(\gamma=1\), 무작위 정책(매 스텝 좌/우 50:50)에서 참값은
\(V^{*}(s) = s/6\)입니다 — "왼쪽 끝(0, 가치 0)에서 오른쪽 끝(6, 가치 1)까지
선형으로 증가"한다는 직관 그대로입니다. 예: \(V^{*}(3)=0.5\).

In [ ]:
import random, math

N_STATES = 7            # 0~6
TERMINAL = {0, 6}
START    = 3

def step(s, a):
    """a: 0=왼쪽, 1=오른쪽. 반환 (보상, 다음 상태, done)."""
    ns = s + (1 if a == 1 else -1)
    if ns == 6:
        return 1.0, 6, True
    if ns == 0:
        return 0.0, 0, True
    return 0.0, ns, False

def gen_episode(seed, start=START):
    random.seed(seed)
    s, states, rewards = start, [start], []
    while True:
        a = random.randrange(2)          # 무작위 정책: 좌/우 50:50
        r, ns, done = step(s, a)
        rewards.append(r); states.append(ns)
        if done: break
        s = ns
    return states, rewards               # states 길이 T+1(마지막 터미널), rewards 길이 T

TRUE_V = [s/6 for s in range(N_STATES)]  # 참값 V*(s)=s/6
print("참값 V*(s)=s/6:", [round(v,3) for v in TRUE_V])
st, rw = gen_episode(0)
print(f"시드0 에피소드: 상태열={st}")
print(f"            보상열={rw}  (에피소드 길이 T={len(rw)})")
print(f"            리턴(보상 합)={sum(rw)}  -> 이 에피소드는 {'오른쪽 끝(6) 도달' if sum(rw)>0 else '왼쪽 끝(0) 도달'}")

## 2. n-step TD (7.1절 코드)

\(n\) 스텝만큼의 실제 보상을 쓰고 그 이후는 추정치 \(V(s_{t+n})\)로
대체하는 목표값 \(G_t^{(n)}\)로 갱신합니다. \(n=1\)이면 TD(0),
\(n\)이 에피소드 길이 이상이면 사실상 MC입니다.

In [ ]:
def n_step_td(n, n_episodes, alpha, gamma=1.0, seed_base=0, start=START):
    V = [0.0]*N_STATES
    for ep in range(n_episodes):
        states, rewards = gen_episode(seed_base+ep, start)
        T = len(rewards)
        for t in range(T):
            G = 0.0
            for k in range(n):
                if t+k < T:
                    G += gamma**k * rewards[t+k]      # n 스텝 실제 보상
            if t+n < T:
                G += gamma**n * V[states[t+n]]        # 나머지 추정치
            V[states[t]] += alpha*(G - V[states[t]])
    return V

def rms_error(V, truth=TRUE_V):
    """비터미널 상태(1~5)의 RMS 오차."""
    return math.sqrt(sum((V[s]-truth[s])**2 for s in range(1,6))/5)

# n=1 (TD(0)) 단일 실행으로 간단히 확인
V1 = n_step_td(1, 10, 0.2)
print("n=1(TD(0)), 10에피소드, 시드0 ->", [round(v,3) for v in V1])
print("참값                       ->", [round(v,3) for v in TRUE_V])
print(f"RMS 오차 = {rms_error(V1):.4f}")

## 3. n을 바꿔가며 U자 곡선 (20회 반복 평균)

\(n\)을 \(1,3,5,10,50\)으로 바꿔가며, 각각 **20개의 서로 다른 시드**로
학습해서 RMS 오차를 **평균**합니다. "20회 반복"이 왜 중요한지는 5절에서
드러납니다. \(\alpha=0.2\), 10 에피소드, \(\gamma=1\).

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

ALPHA, N_EPS, N_REPS = 0.2, 10, 20

def sweep(fn, param_vals, label):
    means = []
    for p in param_vals:
        vals = [rms_error(fn(p, N_EPS, ALPHA, seed_base=s)) for s in range(N_REPS)]
        means.append(sum(vals)/len(vals))
        print(f"{label}={p}:  평균 RMS 오차 = {sum(vals)/len(vals):.4f}")
    return means

ns   = [1, 3, 5, 10, 50]
nmean = sweep(n_step_td, ns, "n")

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(ns, nmean, 'o-', color='#1f77b4', lw=2, ms=8)
best = ns[nmean.index(min(nmean))]
for x,y in zip(ns, nmean):
    ax.annotate(f'{y:.2f}', (x,y), textcoords='offset points', xytext=(0,8),
                ha='center', fontsize=8)
ax.annotate('best', (best, min(nmean)), textcoords='offset points',
            xytext=(0,-16), ha='center', color='green', fontsize=9)
ax.set_xlabel('n (n-step TD)'); ax.set_ylabel('RMS error vs V*=s/6')
ax.set_title('n-step: U-shaped bias-variance tradeoff (10 ep, alpha=0.2, 20-repeat mean)')
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"\n-> n의 U자: 최적 n={best} (평균 {min(nmean):.4f}). 좌극단(n=1)과 우극단(n=50)보다 중간이 낮다.")

## 4. 단 한 번의 실행은 믿지 마라 (단일 시드의 함정)

3절의 **같은 코드**를 **단 한 번**(시드 0)만 돌려서 마지막 \(V\)를 참값과
비교합니다. 이번 실행에서는 우극단인 \(n=50\)이 **가장** 정확할 수
있습니다 — 그런데 20회 평균에서는 \(n=50\)이 양 끝 중 나쁜 쪽이었습니다.
"최적 지점을 찾는 실험"은 여러 시드로 평균내야 합니다.

In [ ]:
print("단일 실행(시드 0, alpha=0.2, 10 에피소드)의 마지막 V vs 참값:")
print(f"{'':4s}" + "".join(f"{'V('+str(s)+')':>9s}" for s in range(1,6)))
for n in [1,5,50]:
    V = n_step_td(n, 10, ALPHA, seed_base=0)
    line = f"n={n:<2d} " + " ".join(f"{V[s]:9.3f}" for s in range(1,6)) + f"   (RMS {rms_error(V):.3f})"
    print(line)
print("true  " + " ".join(f"{v:9.3f}" for v in TRUE_V[1:6]))
print("\n-> 단일 실행에서는 n=50이 RMS 최저(운이 좋아 잘 됨) — 20회 평균의 U자를 단일 실행이 완전히 역전.")
print("   -> 하이퍼파라미터 튜닝은 반드시 여러 시드/반복으로 평균.")

## 5. 적격흔적 TD(\(\lambda\))

n 하나를 고르는 대신, 모든 상태에 "최근에 얼마나 자주, 얼마나 최근에
방문했는가"를 기록한 **흔적** \(e(s)\)를 붙이고, TD 오차 \(\delta_t\)가
발생할 때마다 그 흔적에 비례해 **모든 상태를 동시에** 갱신합니다.

\[ e(s) \leftarrow \gamma\lambda\, e(s) + \mathbb{1}[s=s_t], \qquad
V(s) \leftarrow V(s) + \alpha\,\delta_t\, e(s) \; \text{(모든 s)} \]

갱신 순서는 중요: **① 감쇠(\(\times\gamma\lambda\)) → ② 현재 상태에 +1
→ ③ 모든 s에 \(V \mathrel{+}= \alpha\delta e\)**. 이 순서를 지키면
\(\lambda=0\)이 정확히 TD(0)(\(n=1\))과 같아집니다 — 아래에서
검증합니다.

In [ ]:
def td_lambda(n_episodes, alpha, gamma, lam, seed_base=0, start=START):
    V = [0.0]*N_STATES
    for ep in range(n_episodes):
        e = [0.0]*N_STATES
        random.seed(seed_base+ep); s = start
        while True:
            a = random.randrange(2)
            r, ns, done = step(s, a)
            for i in range(N_STATES):
                e[i] *= gamma*lam                 # ① 감쇠
            e[s] += 1.0                           # ② 현재 상태 +1 (갱신 이전!)
            delta = r + (gamma*V[ns] if not done else 0.0) - V[s]
            for i in range(N_STATES):
                V[i] += alpha*delta*e[i]          # ③ 모든 s 갱신
            s = ns
            if done: break
    return V

# sanity check: lambda=0 이 정확히 n=1(TD(0))과 같아야 한다
for a in [0.1, 0.2]:
    r_n1 = rms_error(n_step_td(1, 10, a))
    r_l0 = rms_error(td_lambda(10, a, 1.0, 0.0))
    ok = abs(r_n1 - r_l0) < 1e-9
    print(f"alpha={a}:  n=1 RMS={r_n1:.4f}   lambda=0 RMS={r_l0:.4f}   일치={ok}")
    assert ok, "lambda=0 가 n=1과 같아야 한다 (갱신 순서 오류)"
print("\n-> lambda=0 이 정확히 TD(0)(n=1)과 일치: 적격흔적 코드의 건성 체크 통과.")
print("   -> lambda=1 이 MC에 가깝다는 것도 아래에서 확인.")

## 6. \(\lambda\)를 바꿔가며 U자 곡선 (20회 반복 평균)

\(\lambda\)를 \(0\)부터 \(1\)까지 바꿔가며, 3절과 **동일하게** 20회
반복 평균합니다. 좌극단 \(\lambda=0\)은 3절의 \(n=1\)과 **완전히
같은 값**(0.3172)이어야 하고, 우극단 \(\lambda=1\)은 MC에 가깝습니다.
바닥은 중간 어딘가(여기서 \(\lambda\approx0.8\))에 있습니다.

In [ ]:
lams  = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
lmean = []
for L in lams:
    vals = [rms_error(td_lambda(N_EPS, ALPHA, 1.0, L, seed_base=s)) for s in range(N_REPS)]
    lmean.append(sum(vals)/len(vals))
    print(f"λ={L}:  평균 RMS 오차 = {sum(vals)/len(vals):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(10,4))
# 재사용: n-panel
ax[0].plot(ns, nmean, 'o-', color='#1f77b4', lw=2, ms=7)
for x,y in zip(ns, nmean): ax[0].annotate(f'{y:.2f}',(x,y),textcoords='offset points',xytext=(0,8),ha='center',fontsize=8)
ax[0].set_title('n-step (n)'); ax[0].set_xlabel('n'); ax[0].grid(alpha=0.3)
ax[1].plot(lams, lmean, 's-', color='#d62728', lw=2, ms=7)
bl = lams[lmean.index(min(lmean))]
for x,y in zip(lams, lmean): ax[1].annotate(f'{y:.2f}',(x,y),textcoords='offset points',xytext=(0,8),ha='center',fontsize=8)
ax[1].annotate('best', (bl, min(lmean)), textcoords='offset points', xytext=(0,-16), ha='center', color='green', fontsize=9)
ax[1].set_title(r'TD($\lambda$)'); ax[1].set_xlabel(r'$\lambda$'); ax[1].grid(alpha=0.3)
ax[0].set_ylabel('RMS error vs V*=s/6')
fig.suptitle('Bias-variance tradeoff, 10 episodes, alpha=0.2, 20-repeat mean', y=1.03)
plt.tight_layout(); plt.show()
print(f"\n-> λ의 U자: 최적 λ={bl} (평균 {min(lmean):.4f}). 좌극단 λ=0 은 n=1과 동일, 우극단 λ=1 은 MC에 가깝다.")

## 7. 흔적이 "어떻게" 모든 상태를 동시에 갱신하는가 (작은 손계산 예)

실제 환경 대신 **방문 순서만 고정**한 4스텝(\(3\to2\to3\to4\), 보상
전부 0)을 만들어, \(\gamma=1\), \(\lambda=0.5\), \(\alpha=0.1\)로
각 스텝의 **흔적 \(e\)와 \(V\) 갱신**을 그대로 출력합니다.
\(\delta_t\)는 실제로 계산된 값을 씁니다. 마지막 스텝(\(t=3\), 상태 4)에서
**하나의** \(\delta_3\)가 **세 개**의 상태(2,3,4)를 갱신하는 것을
확인합니다. (힌트: 상태 3은 두 번 방문돼 흔적이 1.25까지 쌓인다.)

In [ ]:
# 본문 표를 그대로 재현: 방문 순서 3->2->3->4, 보상 전부 0.
# 실제 random walk에서는 중간 보상이 0이라 초기에는 delta=0이 되지만,
# 여기서는 본문의 worked example처럼 각 스텝의 delta가 (0.4, 0.3, 0.5, 0.2)
# 라고 **주어져 있을 때** 흔적 e가 어떻게 퍼지고 V가 어떻게 갱신되는지 보여준다.
# 핵심은 "갱신 메커니즘"(감쇠 -> 현재+1 -> 모든 s에 V+= alpha*delta*e)이다.

seq   = [3, 2, 3, 4]     # s_0..s_3
deltas= [0.4, 0.3, 0.5, 0.2]   # 본문 worked example의 delta_t
alpha, lam, gamma = 0.1, 0.5, 1.0
V = [0.0]*N_STATES; e = [0.0]*N_STATES
print(f"{'t':>2} {'s_t':>4}  {'e(1..4) after decay+1':>24} {'delta':>6}   V 갱신")
for t, s in enumerate(seq):
    for i in range(N_STATES): e[i]*=gamma*lam   # ① 감쇠
    e[s]+=1.0                                   # ② 현재 상태 +1
    delta=deltas[t]
    e_show={str(i+1): round(e[i],3) for i in range(4) if abs(e[i])>1e-9}
    print(f"{t:>2} {s:>4}  {str(e_show):>24} {delta:>6.2f}   ", end="")
    for i in range(N_STATES):                    # ③ 모든 s 갱신
        up=alpha*delta*e[i]
        if abs(up)>1e-9:
            V[i]+=up
            print(f"V({i}){up:+.4f}", end="  ")
    print()
print()
print("-> t=3(상태4)에서 하나의 delta_3=0.2가 V(2), V(3), V(4) 세 상태를 동시에")
print("   갱신(후방 크레딧) - 현재 상태 4(흔적1)가 가장 크고, 직전 3(0.625),")
print("   직전직전 2(0.25) 순으로 작다.")
print("   -> 상태 3은 2번 방문(t=0, t=2) -> 흔적이 쌓여 t=2에서 1.25, 더 많이 갱신.")
assert abs(e[3]-0.625) < 1e-9, "t=3에서 e(3)=0.625여야 한다"
assert abs(V[2]-0.030-0.025-0.005) < 1e-9 and abs(V[3]-0.040-0.015-0.0625-0.0125)<1e-9
print("   -> 본문 표의 수치와 정확히 일치 (검증 통과).")

## 정리

- **n을 고르는 일과 \(\lambda\)를 고르는 일은 같은 U자(편향-분산
  트레이드오프)의 두 모습** — 둘 다 20회 반복 평균으로 바닥(여기서는
  \(n\approx5\), \(\lambda\approx0.8\))을 찾는다.
- **단일 시드/단일 실행으로 최적 지점을 정하면 안 된다** — 하나의 실행은
  운이다 (4절).
- **\(\lambda=0\)=TD(0), \(\lambda=1\)\(\approx\)MC**는 적격흔적
  코드의 건성 체크로 항상 확인 (5절).
- \(\lambda\)는 "n 하나를 고르는 것"을 "모든 n의 가중분포
  \(\propto\lambda^{n-1}\)을 \(\lambda\) 하나로 조절하는 것"으로
  승화시킨 다이얼이다.

이어서 7.3절(Dyna-Q)에서는 경험을 **모델**로 저장해 재사용하는
계획(planning)을 다룹니다.